In [5]:
# Option 1: run the whole script
%run ./CNN_LSTM_parallel.py

# Option 2: import as module
import sys, os
proj_dir = os.getcwd()
if proj_dir not in sys.path:
    sys.path.append(proj_dir)

import CNN_LSTM_parallel


In [6]:
import numpy as np
import pandas as pd
import h5py
import json
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchinfo import summary
import torch.utils.checkpoint as checkpoint
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm, trange
import seaborn as sns
import gc
import time
import warnings
from collections import deque
import psutil
import os

In [7]:
FILE_PATH = '/mnt/sda3/dataset_AMC /radioml2018/versions/2/GOLD_XYZ_OSC.0001_1024.hdf5'
JSON_PATH = '/mnt/sda3/dataset_AMC /radioml2018/versions/2/classes-fixed.json'

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
# Training parameters - OPTIMIZED FOR 4GB VRAM
n_channels = 2
batch_size = 32  # Reduced from 256 to 32 for 4GB VRAM
frame_size = 512
effective_batch = 128  # Reduced from 1024 to 128 for memory efficiency

In [10]:
def dataset_split(data,
                  modulations_classes,
                  modulations,
                  snrs,
                  target_modulations,
                  mode,
                  target_snrs,
                  train_proportion=0.6,
                  valid_proportion=0.2,
                  test_proportion=0.2,
                  seed=48):
    """
    Fixed dataset split function with proper return values
    """
    np.random.seed(seed)
    X_output = []
    Y_output = []
    Z_output = []
    
    # Get valid target modulation indices
    target_modulation_indices = []
    for modu in target_modulations:
        if modu in modulations_classes:
            idx = modulations_classes.index(modu)
            if np.any(modulations == idx):
                target_modulation_indices.append(idx)

    for modu in target_modulation_indices:
        for snr in target_snrs:
            snr_modu_indices = np.where((modulations == modu) & (snrs == snr))[0]
            
            if len(snr_modu_indices) == 0:
                continue
            
            np.random.shuffle(snr_modu_indices)
            num_samples = len(snr_modu_indices)
            train_end = int(train_proportion * num_samples)
            valid_end = int((train_proportion + valid_proportion) * num_samples)

            if mode == 'train':
                indices = snr_modu_indices[:train_end]
            elif mode == 'valid':
                indices = snr_modu_indices[train_end:valid_end]
            elif mode == 'test':
                indices = snr_modu_indices[valid_end:]
            else:
                raise ValueError(f'Unknown mode: {mode}. Valid modes are train, valid, test')

            if len(indices) > 0:
                X_output.append(data[np.sort(indices)])
                Y_output.append(modulations[np.sort(indices)])
                Z_output.append(snrs[np.sort(indices)])

    if len(X_output) == 0:
        return np.array([]), np.array([]), np.array([])
    
    X_array = np.vstack(X_output)
    Y_array = np.concatenate(Y_output)
    Z_array = np.concatenate(Z_output)
    
    # Remap labels to sequential indices
    unique_labels = np.unique(Y_array)
    for index, value in enumerate(unique_labels):
        Y_array[Y_array == value] = index
    
    return X_array, Y_array, Z_array

In [11]:
class RadioML18Dataset(Dataset):
    def __init__(self, mode: str, 
                 target_modulations=None, 
                 train_proportion=0.6,
                 valid_proportion=0.2,
                 test_proportion=0.2,
                 seed=48):
        """
        RadioML 2018.01A Dataset with proper error handling
        
        Args:
            mode: 'train', 'valid', or 'test'
            target_modulations: List of modulation types to include (None = use all available)
            train_proportion: Fraction of data for training
            valid_proportion: Fraction of data for validation  
            test_proportion: Fraction of data for testing
            seed: Random seed for reproducibility
        """
        super(RadioML18Dataset, self).__init__()

        # Load data
        try:
            self.hdf5_file = h5py.File(FILE_PATH, 'r')
            self.modulation_classes = json.load(open(JSON_PATH, 'r'))
        except FileNotFoundError as e:
            print(f"❌ Error: Could not find dataset files.")
            print(f"HDF5 file: {FILE_PATH}")
            print(f"JSON file: {JSON_PATH}")
            raise e
            
        self.X = self.hdf5_file['X']
        self.Y = np.argmax(self.hdf5_file['Y'], axis=1)
        self.Z = self.hdf5_file['Z'][:, 0]

        # Auto-detect available modulations if not specified
        if target_modulations is None:
            available_mods = []
            for mod in self.modulation_classes:
                mod_idx = self.modulation_classes.index(mod)
                if np.any(self.Y == mod_idx):
                    available_mods.append(mod)
            # Use first 10 modulations for manageable training
            self.target_modulations = available_mods[:10]
        else:
            self.target_modulations = target_modulations

        self.target_snrs = np.unique(self.Z)

        print(f"📊 Creating {mode} dataset with {len(self.target_modulations)} modulations")
        print(f"Target modulations: {self.target_modulations}")

        # Split dataset
        self.X_data, self.Y_data, self.Z_data = dataset_split(
            data=self.X,
            modulations_classes=self.modulation_classes,
            modulations=self.Y,
            snrs=self.Z,
            mode=mode,
            train_proportion=train_proportion,
            valid_proportion=valid_proportion,
            test_proportion=test_proportion,
            target_modulations=self.target_modulations,
            target_snrs=self.target_snrs,
            seed=seed
        )

        if len(self.X_data) == 0:
            print(f"❌ No data found for {mode} split!")
            self.hdf5_file.close()
            return

        # Apply I/Q swap correction for AMC compatibility
        print(f"🔧 Applying I/Q swap correction...")
        X_corrected = np.zeros_like(self.X_data)
        X_corrected[:, :, 0] = self.X_data[:, :, 1]  # I = original Q
        X_corrected[:, :, 1] = self.X_data[:, :, 0]  # Q = original I
        self.X_data = X_corrected

        # Store statistics
        self.num_data = self.X_data.shape[0]
        self.num_lbl = len(self.target_modulations)
        self.num_snr = self.target_snrs.shape[0]
        
        print(f"✅ {mode.capitalize()} dataset ready: {self.num_data:,} samples, {self.num_lbl} classes, {self.num_snr} SNRs")
        
        # Close HDF5 file
        self.hdf5_file.close()

    def __len__(self):
        return self.X_data.shape[0] if hasattr(self, 'X_data') else 0

    def __getitem__(self, idx):
        if not hasattr(self, 'X_data') or len(self.X_data) == 0:
            raise IndexError("Dataset is empty")
        
        x, y, z = self.X_data[idx], self.Y_data[idx], self.Z_data[idx]
        # Convert to tensor and transpose for (channels, sequence_length) format
        x = torch.FloatTensor(x).transpose(0, 1)
        y = torch.LongTensor([y]).squeeze()
        z = torch.FloatTensor([z]).squeeze()
        return x, y, z


In [12]:
# Test the working dataset
def test_dataset():
    """Test all dataset splits"""
    print("=" * 60)
    print("TESTING RADIOML DATASET")
    print("=" * 60)
    
    splits = ['train', 'valid', 'test']
    datasets = {}
    
    # Available modulations from your dataset - REDUCED FOR MEMORY EFFICIENCY
    target_mods = ['OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', 
                   '8PSK', '16PSK', '32PSK']  # Reduced from 10 to 8
    
    for split in splits:
        try:
            print(f"\n--- Testing {split.upper()} split ---")
            ds = RadioML18Dataset(
                mode=split,
                target_modulations=target_mods,
                train_proportion=0.6,
                valid_proportion=0.2,
                test_proportion=0.2
            )
            
            if len(ds) > 0:
                # Test data access
                sample_x, sample_y, sample_z = ds[0]
                print(f"Sample shape: {sample_x.shape}")
                print(f"Sample label: {sample_y}")
                print(f"Sample SNR: {sample_z}")
                
                # Test DataLoader with memory optimization
                dataloader = DataLoader(ds, batch_size=16, shuffle=True, num_workers=1)  # Reduced batch size and workers
                batch_x, batch_y, batch_z = next(iter(dataloader))
                print(f"Batch shapes: X={batch_x.shape}, Y={batch_y.shape}, Z={batch_z.shape}")
                
                datasets[split] = ds
                
            else:
                print(f"❌ {split} dataset is empty!")
                
        except Exception as e:
            print(f"❌ Error in {split}: {e}")
    
    print("\n" + "=" * 60)
    print("DATASET TEST COMPLETE")
    print("=" * 60)
    
    # Print summary
    for split, ds in datasets.items():
        print(f"{split.capitalize()}: {len(ds):,} samples")
    
    return datasets

In [ ]:
datasets = test_dataset()
ds = datasets['train']
n_labels = ds.num_lbl
n_snrs = ds.num_snr
frame_size = ds.X_data.shape[1]
        
print(f"\n📋 Updated parameters:")
print(f"n_labels = {n_labels}")
print(f"n_snrs = {n_snrs}")
print(f"frame_size = {frame_size}")
del datasets 

TESTING RADIOML DATASET

--- Testing TRAIN split ---
📊 Creating train dataset with 8 modulations
Target modulations: ['OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK']
🔧 Applying I/Q swap correction...
